# MNIST: размер входа, квантование пикселей и скорость

Сравнение одной MLP-архитектуры с входом 28×28, 16×16 и 16×16 с 16 уровнями яркости. Веса и вычисления — float32.

## Запуск
Python 3.11+. Выполняйте ячейки сверху вниз в новой сессии. Пути к внешним данным задаются в ячейке конфигурации. Исходные данные не подменяются синтетическими.

Зависимости (в Colab при необходимости): `%pip install torch torchvision scikit-learn pandas matplotlib`. После установки перезапустите среду.

## Статус проверки
Обновлено 27 сентября 2026. Старые выводы очищены: они не подтверждают результаты исправленного кода. Подробности локальной проверки — в `VALIDATION.md` в корне репозитория.

## Ограничения
MNIST скачивается при первом запуске. Обновлённое обучение и сравнительный benchmark требуют повторного запуска.


## 1. Настройки


In [ ]:
import copy, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
from sklearn.metrics import ConfusionMatrixDisplay
SEED = 42
EPOCHS = 12
BATCH_SIZE = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT = Path('results/mnist'); OUTPUT.mkdir(parents=True, exist_ok=True)
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def preprocess(images, size, bits=None):
    x = images.float().unsqueeze(1) / 255.0
    if size != 28:
        x = nn.functional.interpolate(x, size=(size,size), mode='bilinear', align_corners=False, antialias=True)
    if bits is not None:
        levels = 2**bits - 1
        x = torch.floor(x.clamp(0,1) * levels) / levels
    return x

class DigitMLP(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.layers = nn.Sequential(nn.Flatten(),nn.Linear(size*size,48),nn.ReLU(),nn.Linear(48,24),nn.ReLU(),nn.Linear(24,10))
    def forward(self,x): return self.layers(x)

def synchronize():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()


## 2. Обучение, контрольные веса и измерение


In [ ]:
def run_variant(name, size, bits, train_raw, test_raw, train_idx, val_idx):
    seed_all()
    x = preprocess(train_raw.data, size, bits)
    xt = preprocess(test_raw.data, size, bits)
    y, yt = train_raw.targets, test_raw.targets
    train = DataLoader(TensorDataset(x[train_idx],y[train_idx]),batch_size=BATCH_SIZE,shuffle=True,generator=torch.Generator().manual_seed(SEED))
    val = DataLoader(TensorDataset(x[val_idx],y[val_idx]),batch_size=512)
    test = DataLoader(TensorDataset(xt,yt),batch_size=512)
    model = DigitMLP(size).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters())
    best_loss, best_state, history = float('inf'), None, []
    synchronize(); start = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train(); total_loss = 0.0
        for bx,by in train:
            bx,by=bx.to(DEVICE),by.to(DEVICE)
            optimizer.zero_grad(); loss=nn.functional.cross_entropy(model(bx),by)
            loss.backward(); optimizer.step(); total_loss += loss.item()*len(by)
        model.eval(); val_loss=0.0
        with torch.no_grad():
            for bx,by in val:
                val_loss += nn.functional.cross_entropy(model(bx.to(DEVICE)),by.to(DEVICE),reduction='sum').item()
        val_loss /= len(val_idx)
        history.append({'epoch':epoch+1,'train_loss':total_loss/len(train_idx),'val_loss':val_loss})
        if val_loss < best_loss: best_loss,best_state=val_loss,copy.deepcopy(model.state_dict())
    synchronize(); train_seconds=time.perf_counter()-start
    model.load_state_dict(best_state); model.eval(); predictions=[]
    with torch.inference_mode():
        for bx,_ in test: predictions.append(model(bx.to(DEVICE)).argmax(1).cpu())
    yp=torch.cat(predictions).numpy()
    # Benchmark is model-only: preprocessing and host-to-device transfer are excluded.
    batch=xt[:512].to(DEVICE); times=[]
    with torch.inference_mode():
        for _ in range(10): model(batch)
        for _ in range(30):
            synchronize(); start=time.perf_counter(); model(batch); synchronize()
            times.append(time.perf_counter()-start)
    median_s=float(np.median(times))
    ConfusionMatrixDisplay.from_predictions(yt.numpy(),yp,labels=range(10))
    plt.title(name+' — весь test'); plt.show()
    torch.save({'state_dict':{k:v.cpu() for k,v in best_state.items()},'input_size':size,'input_bits':bits,'seed':SEED},OUTPUT/f'{name}.pth')
    pd.DataFrame(history).to_csv(OUTPUT/f'{name}_history.csv',index=False)
    return {'variant':name,'accuracy':float((yp==yt.numpy()).mean()),'parameters':sum(p.numel() for p in model.parameters()),
            'train_seconds':train_seconds,'inference_images_sec':len(batch)/median_s,'batch_size':len(batch),'device':str(DEVICE)}


## 3. Данные и сравнение


In [ ]:
train_raw=MNIST('data',train=True,download=True)
test_raw=MNIST('data',train=False,download=True)
indices=torch.randperm(len(train_raw),generator=torch.Generator().manual_seed(SEED))
cut=int(0.85*len(indices)); train_idx,val_idx=indices[:cut],indices[cut:]
rows=[run_variant(name,size,bits,train_raw,test_raw,train_idx,val_idx)
      for name,size,bits in [('baseline_28',28,None),('resized_16',16,None),('quantized_input_16',16,4)]]
results=pd.DataFrame(rows); display(results)
results.to_csv(OUTPUT/'comparison.csv',index=False)


## Выводы
Сравните baseline→resized, затем resized→quantized_input: это отделяет изменение разрешения от квантования. Значения с 16 уровнями хранятся как float32; здесь нет INT4-инференса или упаковки пикселей в 4 бита. Скорость — медиана 30 замеров после прогрева. Один seed не доказывает статистическую значимость различий.
